In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
df = pd.read_csv("data/enriched_crop_data.csv")
df

,State,District,Season,Year,Crop,Area_Ha,Yield_QHa,Start_Year,District_norm,State_norm,...,longitude,soil_ph,soil_oc,clay_pct,sand_pct,cec_cmol,avg_temp,humidity_avg,rain_total,solar_avg
0,Chandigarh,Chandigarh,Whole Year,2015 - 2016,Potato,10.0,155.0,2015,chandigarh,chandigarh,...,76.767333,NaN,NaN,NaN,NaN,NaN,23.899342,50.669836,1130.12,18.550904
1,Chandigarh,Chandigarh,Kharif,2015 - 2016,Rice,20.0,51.5,2015,chandigarh,chandigarh,...,76.767333,NaN,NaN,NaN,NaN,NaN,29.421639,62.886967,840.64,21.009426
2,Chandigarh,Chandigarh,Kharif,2016 - 2017,Rice,8.0,52.5,2016,chandigarh,chandigarh,...,76.767333,NaN,NaN,NaN,NaN,NaN,30.185328,63.451721,712.46,20.838361
3,Chandigarh,Chandigarh,Kharif,2017 - 2018,Rice,10.0,55.0,2017,chandigarh,chandigarh,...,76.767333,NaN,NaN,NaN,NaN,NaN,29.200328,67.663607,991.37,20.418279
4,Chandigarh,Chandigarh,Kharif,2018 - 2019,Rice,10.0,53.0,2018,chandigarh,chandigarh,...,76.767333,NaN,NaN,NaN,NaN,NaN,28.788443,72.857705,1270.35,18.903033
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58948,West Bengal,Purulia,Rabi,2022 - 2023,Arhar/Tur,454.0,12.6,2022,puruliya,west bengal,...,86.295116,6.4,24.6,31.8,32.3,23.6,20.858626,63.868352,190.68,15.362308
58949,West Bengal,Purulia,Rabi,2022 - 2023,Horse-gram,1250.0,5.1,2022,puruliya,west bengal,...,86.295116,6.4,24.6,31.8,32.3,23.6,20.858626,63.868352,190.68,15.362308
58950,West Bengal,Purulia,Kharif,2022 - 2023,Niger seed,354.0,4.1,2022,puruliya,west bengal,...,86.295116,6.4,24.6,31.8,32.3,23.6,28.273852,81.411557,1027.63,16.674262
58951,West Bengal,Purulia,Kharif,2022 - 2023,Bajra,70.0,4.3,2022,puruliya,west bengal,...,86.295116,6.4,24.6,31.8,32.3,23.6,28.273852,81.411557,1027.63,16.674262


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58953 entries, 0 to 58952
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   State          58953 non-null  object 
 1   District       58953 non-null  object 
 2   Season         58953 non-null  object 
 3   Year           58953 non-null  object 
 4   Crop           58953 non-null  object 
 5   Area_Ha        58953 non-null  float64
 6   Yield_QHa      58953 non-null  float64
 7   Start_Year     58953 non-null  int64  
 8   District_norm  58953 non-null  object 
 9   State_norm     58953 non-null  object 
 10  latitude       58953 non-null  float64
 11  longitude      58953 non-null  float64
 12  soil_ph        56447 non-null  float64
 13  soil_oc        56447 non-null  float64
 14  clay_pct       56447 non-null  float64
 15  sand_pct       56447 non-null  float64
 16  cec_cmol       56447 non-null  float64
 17  avg_temp       58953 non-null  float64
 18  humidi

In [4]:
crop_mean = {}
crop_count = {}
for crop in df["Crop"].unique():
    crop_mean[crop] = float(round(df[df["Crop"] == crop]["Yield_QHa"].mean(), 2))
    crop_count[crop] = int(df[df["Crop"] == crop].shape[0])

crop_mean

{'Potato': 206.07,
 'Rice': 26.04,
 'Arhar/Tur': 9.64,
 'Bajra': 14.52,
 'Castor seed': 12.66,
 'Cotton(lint)': 22.43,
 'Gram': 10.39,
 'Groundnut': 16.17,
 'Guar seed': 8.09,
 'Moong(Green Gram)': 4.73,
 'Moth': 4.53,
 'Other Kharif pulses': 6.41,
 'Rapeseed &Mustard': 11.51,
 'Sesamum': 6.29,
 'Tobacco': 24.76,
 'Urad': 5.67,
 'Wheat': 30.12,
 'Onion': 159.68,
 'Soyabean': 11.42,
 'Dry chillies': 12.38,
 'Garlic': 57.94,
 'Maize': 25.37,
 'Other Rabi pulses': 7.59,
 'Jowar': 11.1,
 'Banana': 348.93,
 'Sugarcane': 702.37,
 'Other Cereals': 6.63,
 'Small millets': 7.26,
 'Ragi': 13.03,
 'Ginger': 117.32,
 'Masoor': 8.57,
 'Peas & beans (Pulses)': 21.62,
 'Sunflower': 7.93,
 'Turmeric': 37.52,
 'Coriander': 8.32,
 'other oilseeds': 5.95,
 'Barley': 26.98,
 'Horse-gram': 5.39,
 'Sannhamp': 6.36,
 'Sweet potato': 127.85,
 'Cowpea(Lobia)': 4.07,
 'Linseed': 4.67,
 'Coconut': 82184.86,
 'Safflower': 6.59,
 'Niger seed': 2.42,
 'Cashewnut': 11.25,
 'Tapioca': 251.02,
 'Arecanut': 64.84,
 'Ca

In [5]:
print("Mean Yield per Crop:")
for crop, mean_yield in crop_mean.items():
    print(f"{crop}: {crop_count[crop]} samples: {mean_yield} Q/Ha")

Mean Yield per Crop:
Potato: 1906 samples: 206.07 Q/Ha
Rice: 2265 samples: 26.04 Q/Ha
Arhar/Tur: 2096 samples: 9.64 Q/Ha
Bajra: 1742 samples: 14.52 Q/Ha
Castor seed: 697 samples: 12.66 Q/Ha
Cotton(lint): 1376 samples: 22.43 Q/Ha
Gram: 2145 samples: 10.39 Q/Ha
Groundnut: 1744 samples: 16.17 Q/Ha
Guar seed: 1028 samples: 8.09 Q/Ha
Moong(Green Gram): 2212 samples: 4.73 Q/Ha
Moth: 266 samples: 4.53 Q/Ha
Other Kharif pulses: 891 samples: 6.41 Q/Ha
Rapeseed &Mustard: 2025 samples: 11.51 Q/Ha
Sesamum: 2080 samples: 6.29 Q/Ha
Tobacco: 650 samples: 24.76 Q/Ha
Urad: 2255 samples: 5.67 Q/Ha
Wheat: 2361 samples: 30.12 Q/Ha
Onion: 1872 samples: 159.68 Q/Ha
Soyabean: 847 samples: 11.42 Q/Ha
Dry chillies: 1368 samples: 12.38 Q/Ha
Garlic: 1394 samples: 57.94 Q/Ha
Maize: 3308 samples: 25.37 Q/Ha
Other Rabi pulses: 706 samples: 7.59 Q/Ha
Jowar: 2073 samples: 11.1 Q/Ha
Banana: 254 samples: 348.93 Q/Ha
Sugarcane: 2087 samples: 702.37 Q/Ha
Other Cereals: 544 samples: 6.63 Q/Ha
Small millets: 821 samples: 7